In [7]:
import pandas as pd
import os
import re

FILE_PATH = "cloud_database.csv"

In [8]:
def initialize_database():
    if not os.path.exists(FILE_PATH):
        df = pd.DataFrame(columns=["ID", "Name", "Email"])
        df.to_csv(FILE_PATH, index=False)
        print("✅ Database created successfully!")
    else:
        print("📂 Database already exists.")

initialize_database()

📂 Database already exists.


In [9]:
def load_database():
    return pd.read_csv(FILE_PATH)

db = load_database()
db

,ID,Name,Email
0,1,Ali,ali@gmail.com
1,2,John,john@gmail.com
2,3,Ali,ali@gmail.com


In [10]:
def is_valid_email(email):
    pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
    return bool(re.match(pattern, str(email)))

In [11]:
def clean_and_add_data(new_data):
    global db

    try:
        new_df = pd.DataFrame(new_data)

        # Check required columns
        required_columns = {"ID", "Name", "Email"}
        if not required_columns.issubset(new_df.columns):
            print("❌ Error: Missing required fields!")
            return db

        # Remove duplicates inside new data
        new_df = new_df.drop_duplicates()

        # Validate emails
        new_df = new_df[new_df["Email"].apply(is_valid_email)]

        if new_df.empty:
            print("⚠️ No valid data after validation.")
            return db

        # Remove duplicates with database
        unique_data = new_df[~new_df["Email"].isin(db["Email"])]

        if unique_data.empty:
            print("⚠️ All entries are duplicates.")
            return db

        # Append unique data
        db = pd.concat([db, unique_data], ignore_index=True)

        # Save to file
        db.to_csv(FILE_PATH, index=False)

        print(f"✅ {len(unique_data)} new records added successfully!")

    except Exception as e:
        print(f"❌ Error occurred: {e}")

    return db

In [12]:
new_entries = [
    {"ID": 1, "Name": "Ali", "Email": "ali@gmail.com"},
    {"ID": 2, "Name": "John", "Email": "john@gmail.com"},
    {"ID": 3, "Name": "Ali", "Email": "ali@gmail.com"},  # duplicate
    {"ID": 4, "Name": "Invalid", "Email": "invalidemail"}  # invalid
]

updated_db = clean_and_add_data(new_entries)
updated_db

⚠️ All entries are duplicates.


,ID,Name,Email
0,1,Ali,ali@gmail.com
1,2,John,john@gmail.com
2,3,Ali,ali@gmail.com
